# 13 — Beta-Neutral Momentum

Primary test: preserve the existing raw momentum portfolio and hedge its ex-ante net market beta. The comparator uses the same beta-eligible holdings so beta coverage is separated from neutralisation.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.daily_data import read_daily_stock, read_daily_market, read_fama_french_daily_rf, align_market_and_rf
from src.beta import BetaConfig, estimate_monthly_capm_beta
from src.beta_neutral import build_beta_neutral_momentum, ex_post_market_beta
from src.inference import primary_and_robustness_hac


## 1. Rebuild the frozen baseline signal


In [ ]:
MONTHLY = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
DAILY_STOCK = ROOT / 'data/raw/daily_stock_naq_1998_2025_prod_v1.csv.gz'
DAILY_MARKET = ROOT / 'data/raw/daily_crsp_vw_market_1998_2025_v1.csv.gz'
FF_DAILY = ROOT / 'data/raw/F-F_Research_Data_Factors_daily.csv'

clean, _ = build_clean_monthly_panel(MONTHLY, CleaningConfig())
history = build_monthly_history_panel(MONTHLY)
signal = attach_momentum_to_formation_universe(clean, history)
signal = assign_signal_quintiles(signal)

print('Formation rows:', len(signal))
print('Valid momentum:', signal['Momentum_12_2'].notna().sum())


## 2. Re-estimate ex-ante beta under the locked specification


In [ ]:
daily_stock = read_daily_stock(DAILY_STOCK)
market = read_daily_market(DAILY_MARKET)
rf = read_fama_french_daily_rf(FF_DAILY)
market_rf, _ = align_market_and_rf(market, rf)

beta, beta_audit = estimate_monthly_capm_beta(
    clean[['PERMNO','MthCalDt']],
    daily_stock,
    market_rf,
    config=BetaConfig(lookback_days=252, skip_days=5, min_obs=126),
)


## 3. Construct beta-eligible raw and beta-neutral spreads


In [ ]:
panel = build_beta_neutral_momentum(signal, beta, market_rf)
panel.head()


## 4. Exposure and coverage diagnostics


In [ ]:
panel[['Beta_Q1','Beta_Q5','Beta_LS','Q1_BetaEligibleCountShare','Q5_BetaEligibleCountShare','Q1_BetaEligibleCapShare','Q5_BetaEligibleCapShare']].describe(percentiles=[0.01,0.05,0.5,0.95,0.99])


In [ ]:
print('Months with non-positive Q1 beta:', int((panel['Beta_Q1'] <= 0).sum()))
print('Months with non-positive Q5 beta:', int((panel['Beta_Q5'] <= 0).sum()))
print('Months with scaled-neutral robustness available:', panel['ScaledNeutralSpread'].notna().sum())
print('Max absolute ex-ante beta after primary hedge:', panel['ExAnteBetaAfterHedge'].abs().max())


## 5. Mean returns and decomposition


In [ ]:
cols = ['OriginalSpread','EligibleSpread','BetaNeutralSpread','CoverageEffect','NeutralisationEffect','TotalVsOriginal','ScaledNeutralSpread','ScaledNeutralisationEffect']
panel[cols].agg(['count','mean','std']).T


## 6. Primary inference: neutralisation effect


In [ ]:
primary_and_robustness_hac(panel['NeutralisationEffect'])


## 7. Coverage-effect inference


In [ ]:
primary_and_robustness_hac(panel['CoverageEffect'])


## 8. Long/short rescaling robustness


In [ ]:
primary_and_robustness_hac(panel['ScaledNeutralisationEffect'])


## 9. Ex-post realised monthly market beta


In [ ]:
expost = pd.DataFrame({
    'OriginalRaw': ex_post_market_beta(panel, 'OriginalSpread', hac_lag=6),
    'BetaEligibleRaw': ex_post_market_beta(panel, 'EligibleSpread', hac_lag=6),
    'BetaNeutral': ex_post_market_beta(panel, 'BetaNeutralSpread', hac_lag=6),
}).T
expost


## Interpretation gate

Primary interpretation uses **NeutralisationEffect = BetaNeutralSpread - EligibleSpread**. CoverageEffect is separate. The leg-rescaling construction is robustness only.
